In [1]:
import pandas as pd
import json 
import pickle
import pandas as pd

from jestr.utils.eval import borda_count, get_target, convert_rank_to_hit_rates

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

/home/etienne/miniforge3/envs/jestr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1 Input data

## 1.1 JESTR input data 
1. spectra_metadata_test.tsv
2. identifier_to_candidates_test.json, if not specified will be retrieve from huggingace
For running JESTR on complete MassSpecGym run unlabel.ipynb, will create spectra_metadata.tsv

In [17]:
spectra = pd.read_csv("tests/data/spectra_metadata_test.tsv", sep="\t")
display(spectra.head(1))

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge
0,MassSpecGymID0000201,"83.0491,123.0417,134.0964,141.0509,156.0781,202.0836,216.1383,244.1308,266.1135,284.1254,302.136,384.1765","0.16216216216216217,0.04704704704704705,1.0,0.043043043043043044,0.2032032032032032,0.30430430430430433,0.2032032032032032,0.11911911911911911,0.6256256256256256,0.2782782782782783,0.04604604604604605,0.4064064064064064",CC(C)[C@@H]1C(=O)N([C@H](C(=O)O[C@@H](C(=O)N([C@H](C(=O)O[C@@H](C(=O)N([C@H](C(=O)O1)CC2=CC=CC=C2)C)C(C)C)CC3=CC=CC=C3)C)C(C)C)CC4=CC=CC=C4)C,GYSCAQFHASJXRS,C45H57N3O9,C45H57N3NaO9,783.408982,806.3982,[M+Na]+,Orbitrap,50.0,test,False


## 1.2 JESTR with FAISS input data: 
1. unlabeled_spectra_test.json: list of spectra with metadata
2. list_of_lists_of_candidates_test.json: a list of candidate sets for each spectrum

Both input lists have a certain order so the first spectrum correspons to the first candidate list etc.

Example input files are provided in tests/data/ to run JESTR with FAISS on the full MassSpecGym dataset run unlabel-ipynb,
this will store the complete files in the data folder.


In [9]:
with open("tests/data/unlabeled_spectra_test.json", "r") as f:
    spectra = json.load(f)
display(spectra[0])

with open("tests/data/list_of_lists_of_candidates_test.json", "r") as f:
    candidates = json.load(f)
    display(candidates[0])

{'identifier': 'MassSpecGymID0000201',
 'precursor_formula': 'C45H57N3NaO9',
 'parent_mass': 783.4089819999999,
 'precursor_mz': 806.3982,
 'peaks_mz': [],
 'adduct': '[M+Na]+',
 'instrument_type': 'Orbitrap',
 'collision_energy': 50.0,
 'simulation_challenge': False,
 'peaks_json': [[83.0491, 0.16216216216216217],
  [123.0417, 0.04704704704704705],
  [134.0964, 1.0],
  [141.0509, 0.043043043043043044],
  [156.0781, 0.2032032032032032],
  [202.0836, 0.30430430430430433],
  [216.1383, 0.2032032032032032],
  [244.1308, 0.11911911911911911],
  [266.1135, 0.6256256256256256],
  [284.1254, 0.2782782782782783],
  [302.136, 0.04604604604604605],
  [384.1765, 0.4064064064064064]]}

['CC(C)[C@@H]1C(=O)N([C@H](C(=O)O[C@@H](C(=O)N([C@H](C(=O)O[C@@H](C(=O)N([C@H](C(=O)O1)CC2=CC=CC=C2)C)C(C)C)CC3=CC=CC=C3)C)C(C)C)CC4=CC=CC=C4)C',
 'CC(C)C1C(=O)OC(Cc2ccccc2)C(=O)N(C)C(C(C)C)C(=O)OC(Cc2ccccc2)C(=O)N(C)C(C(C)C)C(=O)OC(Cc2ccccc2)C(=O)N1C',
 'COC(=O)[C@@H]1CCCN1C(=O)[C@H](C)NC(=O)[C@H](CC(C)C)NC(=O)/C=C/[C@H]1O[C@@H](C)[C@@H](OCc2ccccc2)[C@@H](OCc2ccccc2)[C@@H]1OCc1ccccc1',
 'C=CCO[C@@]12Oc3ccc(OC(=O)NCC)cc3[C@H]3[C@H](CCCCO)[C@@H](CCCCO)C=C(C(=NOCC)C[C@@H]1N(Cc1cccc4ccccc14)C(=O)OC)[C@H]32',
 'COCCCN1C(=O)COc2ccc(N(C(=O)[C@H]3CN(C(=O)OC(C)(C)C)CC[C@@H]3c3cccc(-c4cccc(OCCOC5CCCCO5)c4)c3)C3CC3)cc21',
 'CC(C)[C@H]1C(=O)O[C@H](Cc2ccccc2)C(=O)N(C)[C@@H](C(C)C)C(=O)O[C@H](Cc2ccccc2)C(=O)N(C)[C@@H](C(C)C)C(=O)O[C@H](Cc2ccccc2)C(=O)N1C',
 'C=CCO[C@@]12Oc3ccc(OC(=O)NCC)cc3[C@H]3[C@H](CCCCO)[C@@H](CCCCO)C=C(C(=NOC)C[C@@H]1N(Cc1cccc4ccccc14)C(=O)OCC)[C@H]32',
 'C=CCOC12Oc3ccc(OC(=O)NCC)cc3C3C(CCCCO)C(CCCCO)C=C(C(=NOCC)CC1N(Cc1cccc4ccccc14)C(=O)OC)C32',
 'CC(C)[C@H](CC(=O)[C@H](CCCCO

# 2 Run program

## 2.1 Running JESTR
```bash
cd jestr
python -m inference
```

## 2.2 Running JESTR with FAISS
```bash
cd jestr
python -m faiss_precompute
python -m faiss_inference
```

In [3]:
def display_hit_rates(dataframe):
    dataframe["rank"] = dataframe.apply(
        lambda row: borda_count(row["candidates"], [row["scores"]], get_target(row["candidates"], row["labels"])),
        axis=1,
    )

    hit_rate_cols = dataframe.apply(
        lambda row: convert_rank_to_hit_rates(row, "rank", top_k=[1, 5, 20]),
        axis=1,
    )
    dataframe = pd.concat([dataframe, hit_rate_cols], axis=1)

    hit_rate_summary = dataframe[["rank-hit_rate@1", "rank-hit_rate@5", "rank-hit_rate@20"]].mean().to_dict()
    ranks = dataframe["rank"].tolist()
    mean_hit_rates = dataframe[["rank-hit_rate@1", "rank-hit_rate@5", "rank-hit_rate@20"]].mean()
    return mean_hit_rates

# 3 Output

## 3.1 FAISS Pipeline:

In [ ]:
with open("experiments/20260321_FAISS_sample_run_1/result_unlabeled_spectra_test.pkl", "rb") as f:
    result_faiss = pickle.load(f)
result_faiss_df = pd.DataFrame(result_faiss)
result_faiss_df.head(1)

,identifier,candidates,scores,labels
0,MassSpecGymID0000201,"[CCCCCCCC(=O)Oc1ccc(-c2nc(-c3ccc(OC(=O)CCCCCCC)cc3O)nc(-c3ccc(OC(=O)CCCCCCC)cc3O)n2)c(O)c1, CCCCC(CC)C(=O)Oc1ccc(-c2nc(-c3ccc(OC(=O)C(CC)CCCC)cc3O)nc(-c3ccc(OC(=O)C(CC)CCCC)cc3O)n2)c(O)c1, COCCCN1C(=O)COc2ccc(N(C(=O)[C@H]3CN(C(=O)OC(C)(C)C)CC[C@@H]3c3cccc(-c4ccc(OCCOC5CCCCO5)cc4)c3)C3CC3)cc21, CCCCCCCC(=O)Oc1ccc(-n2c(=O)n(-c3ccc(OC(=O)CCCCCCC)cc3)c(=O)n(-c3ccc(OC(=O)CCCCCCC)cc3)c2=O)cc1, C=CCOC12Oc3ccc(OC(=O)NCC)cc3C3C(CCCCO)C(CCCCO)C=C(C(=NOC)CC1N(Cc1cccc4ccccc14)C(=O)OCC)C32, C=CCOC12Oc3ccc(OC(=O)NCC)cc3C3C(CCCCO)C(CCCCO)C=C(C(=NOCC)CC1N(Cc1cccc4ccccc14)C(=O)OC)C32, CC(=O)O[C@@H](C)/C=C\C(=O)N[C@@H]1C[C@H](C)[C@H](C/C=C(C)/C=C/[C@@H]2C[C@]3(CO3)C[C@@H](CNC(=O)CCNC(=O)OCC3c4ccccc4-c4ccccc43)O2)O[C@@H]1C, CC(=O)O[C@@H](C)/C=C\C(=O)N[C@@H]1C[C@H](C)[C@H](C/C=C(C)/C=C/[C@@H]2C[C@]3(CO3)C[C@@H](CC(=O)NCCNC(=O)OCC3c4ccccc4-c4ccccc43)O2)O[C@@H]1C, COCCCN1C(=O)COc2ccc(N(C(=O)[C@H]3CN(C(=O)OC(C)(C)C)CC[C@@H]3c3cccc(-c4cccc(OCCOC5CCCCO5)c4)c3)C3CC3)cc21, COC(=O)/C(C)=C\CC1(O)C(=O)C2CC(C(C)C)C13Oc1c(CC=C(C)C)c4c(c(OCNCCO)c1C(=O)C3C2C(C#N)C#N)C=CC(C)(CCC=C(C)C)O4, COC(=O)[C@@H]1CCCN1C(=O)[C@H](C)NC(=O)[C@H](CC(C)C)NC(=O)/C=C/[C@H]1O[C@@H](C)[C@@H](OCc2ccccc2)[C@@H](OCc2ccccc2)[C@@H]1OCc1ccccc1, O=C(O)C[C@H](NC(=O)C[C@H](NC(=O)C[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)C1CCCCC1)C(=O)C1CCCCC1)C(=O)C1CCCCC1, CC(C)[C@H](CC(=O)[C@H](CCCCOC(=O)c1ccccc1)NC(=O)OC(C)(C)C)C(=O)N[C@@H](Cc1ccccc1)C(=O)N1CCC[C@@H]1C(=O)OCc1ccccc1, CC(C)[C@H](CC(=O)[C@@H](CCCCOC(=O)c1ccccc1)NC(=O)OC(C)(C)C)C(=O)N[C@@H](Cc1ccccc1)C(=O)N1CCC[C@@H]1C(=O)OCc1ccccc1, CCC(C)C1OC(=O)C(Cc2ccccc2)N(C)C(=O)C(C(C)C)OC(=O)C(Cc2ccccc2)N(C)C(=O)C(C(C)C)OC(=O)C(Cc2ccccc2)NC1=O, C=CC(CCN(C(=O)OCC1c2ccccc2-c2ccccc21)[C@@H](CCCCNC(=O)OCc1ccccc1)C(=O)OC)[C@H]1COC(C)(C)N1C(=O)OC(C)(C)C, C[C@H](CCC(=O)ON1C(=O)CCC1=O)[C@H]1CC[C@H]2[C@@H]3CC[C@@H]4C[C@@](O)(NC(=O)CNC(=O)OCC5c6ccccc6-c6ccccc65)CC[C@]4(C)[C@H]3C[C@H](O)[C@]12C, C=CCO[C@@]12Oc3ccc(OC(=O)NCC)cc3[C@H]3[C@H](CCCCO)[C@@H](CCCCO)C=C(C(=NOCC)C[C@@H]1N(Cc1cccc4ccccc14)C(=O)OC)[C@H]32, C=CCO[C@@]12Oc3ccc(OC(=O)NCC)cc3[C@H]3[C@H](CCCCO)[C@@H](CCCCO)C=C(C(=NOC)C[C@@H]1N(Cc1cccc4ccccc14)C(=O)OCC)[C@H]32, CCC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@H](C(C)C)OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@H](C(C)C)OC(=O)[C@H](Cc2ccccc2)NC1=O]","[0.27783945202827454, 0.2568430006504059, 0.12162548303604126, 0.11344578862190247, 0.09720828384160995, 0.09243404120206833, 0.07877541333436966, 0.07485778629779816, 0.07162836194038391, 0.05109420418739319, 0.03140075504779816, 0.0118465106934309, -0.00536822434514761, -0.005509334150701761, -0.04881744086742401, -0.05484038591384888, -0.08987842500209808, -0.09370101988315582, -0.10384807735681534, -0.11205046623945236]","[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]"


In [5]:
display_hit_rates(result_faiss_df)

rank-hit_rate@1     0.000000
rank-hit_rate@5     0.000000
rank-hit_rate@20    0.333333
dtype: float64

## 3.2 Regular Pipeline:

In [ ]:
with open("test_results/20260713_JESTR_sample_run/result_spectra_metadata_test.pkl", "rb") as f:   
    result_faiss = pickle.load(f)
result_faiss_df = pd.DataFrame(result_faiss)
result_faiss_df.head(1)

,identifier,candidates,scores,labels
0,MassSpecGymID0000201,"[CCCCCCCC(=O)Oc1ccc(-c2nc(-c3ccc(OC(=O)CCCCCCC)cc3O)nc(-c3ccc(OC(=O)CCCCCCC)cc3O)n2)c(O)c1, CCCCC(CC)C(=O)Oc1ccc(-c2nc(-c3ccc(OC(=O)C(CC)CCCC)cc3O)nc(-c3ccc(OC(=O)C(CC)CCCC)cc3O)n2)c(O)c1, COCCCN1C(=O)COc2ccc(N(C(=O)[C@H]3CN(C(=O)OC(C)(C)C)CC[C@@H]3c3cccc(-c4ccc(OCCOC5CCCCO5)cc4)c3)C3CC3)cc21, CCCCCCCC(=O)Oc1ccc(-n2c(=O)n(-c3ccc(OC(=O)CCCCCCC)cc3)c(=O)n(-c3ccc(OC(=O)CCCCCCC)cc3)c2=O)cc1, C=CCOC12Oc3ccc(OC(=O)NCC)cc3C3C(CCCCO)C(CCCCO)C=C(C(=NOC)CC1N(Cc1cccc4ccccc14)C(=O)OCC)C32, C=CCOC12Oc3ccc(OC(=O)NCC)cc3C3C(CCCCO)C(CCCCO)C=C(C(=NOCC)CC1N(Cc1cccc4ccccc14)C(=O)OC)C32, CC(=O)O[C@@H](C)/C=C\C(=O)N[C@@H]1C[C@H](C)[C@H](C/C=C(C)/C=C/[C@@H]2C[C@]3(CO3)C[C@@H](CNC(=O)CCNC(=O)OCC3c4ccccc4-c4ccccc43)O2)O[C@@H]1C, CC(=O)O[C@@H](C)/C=C\C(=O)N[C@@H]1C[C@H](C)[C@H](C/C=C(C)/C=C/[C@@H]2C[C@]3(CO3)C[C@@H](CC(=O)NCCNC(=O)OCC3c4ccccc4-c4ccccc43)O2)O[C@@H]1C, COCCCN1C(=O)COc2ccc(N(C(=O)[C@H]3CN(C(=O)OC(C)(C)C)CC[C@@H]3c3cccc(-c4cccc(OCCOC5CCCCO5)c4)c3)C3CC3)cc21, COC(=O)/C(C)=C\CC1(O)C(=O)C2CC(C(C)C)C13Oc1c(CC=C(C)C)c4c(c(OCNCCO)c1C(=O)C3C2C(C#N)C#N)C=CC(C)(CCC=C(C)C)O4, COC(=O)[C@@H]1CCCN1C(=O)[C@H](C)NC(=O)[C@H](CC(C)C)NC(=O)/C=C/[C@H]1O[C@@H](C)[C@@H](OCc2ccccc2)[C@@H](OCc2ccccc2)[C@@H]1OCc1ccccc1, O=C(O)C[C@H](NC(=O)C[C@H](NC(=O)C[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)C1CCCCC1)C(=O)C1CCCCC1)C(=O)C1CCCCC1, CC(C)[C@H](CC(=O)[C@H](CCCCOC(=O)c1ccccc1)NC(=O)OC(C)(C)C)C(=O)N[C@@H](Cc1ccccc1)C(=O)N1CCC[C@@H]1C(=O)OCc1ccccc1, CC(C)[C@H](CC(=O)[C@@H](CCCCOC(=O)c1ccccc1)NC(=O)OC(C)(C)C)C(=O)N[C@@H](Cc1ccccc1)C(=O)N1CCC[C@@H]1C(=O)OCc1ccccc1, CCC(C)C1OC(=O)C(Cc2ccccc2)N(C)C(=O)C(C(C)C)OC(=O)C(Cc2ccccc2)N(C)C(=O)C(C(C)C)OC(=O)C(Cc2ccccc2)NC1=O, C=CC(CCN(C(=O)OCC1c2ccccc2-c2ccccc21)[C@@H](CCCCNC(=O)OCc1ccccc1)C(=O)OC)[C@H]1COC(C)(C)N1C(=O)OC(C)(C)C, C[C@H](CCC(=O)ON1C(=O)CCC1=O)[C@H]1CC[C@H]2[C@@H]3CC[C@@H]4C[C@@](O)(NC(=O)CNC(=O)OCC5c6ccccc6-c6ccccc65)CC[C@]4(C)[C@H]3C[C@H](O)[C@]12C, C=CCO[C@@]12Oc3ccc(OC(=O)NCC)cc3[C@H]3[C@H](CCCCO)[C@@H](CCCCO)C=C(C(=NOCC)C[C@@H]1N(Cc1cccc4ccccc14)C(=O)OC)[C@H]32, C=CCO[C@@]12Oc3ccc(OC(=O)NCC)cc3[C@H]3[C@H](CCCCO)[C@@H](CCCCO)C=C(C(=NOC)C[C@@H]1N(Cc1cccc4ccccc14)C(=O)OCC)[C@H]32, CCC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@H](C(C)C)OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@H](C(C)C)OC(=O)[C@H](Cc2ccccc2)NC1=O]","[0.27783945202827454, 0.2568430006504059, 0.12162548303604126, 0.11344578862190247, 0.09720828384160995, 0.09243404120206833, 0.07877541333436966, 0.07485778629779816, 0.07162836194038391, 0.05109420418739319, 0.03140075504779816, 0.0118465106934309, -0.00536822434514761, -0.005509334150701761, -0.04881744086742401, -0.05484038591384888, -0.08987842500209808, -0.09370101988315582, -0.10384807735681534, -0.11205046623945236]","[False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False]"


In [15]:
display_hit_rates(result_faiss_df)

rank-hit_rate@1     0.000000
rank-hit_rate@5     0.000000
rank-hit_rate@20    0.333333
dtype: float64